# ComplaintLens: Routing and Theme Discovery in Consumer Finance Complaints

**Text Mining & Search, University of Milano-Bicocca, A.Y. 2026/27**
Team: **Abdul Sami** and **Fatima** · Instructors: Prof. Marco Viviani, Stefano Molari

Banks and other financial companies receive thousands of written complaints every month, and someone has to read them, send them to the right team and understand what keeps going wrong. We use real complaints published by the US **Consumer Financial Protection Bureau (CFPB)** to study two questions:

1. **Task 1, complaint routing (text classification):** can we send a complaint to the right product team automatically?
2. **Task 2, complaint themes (topic modeling):** what are people actually complaining about, without reading every complaint?

For both tasks we compare a **sparse baseline** with a **dense, embedding-based alternative**, with separate train / validation / test data.

| Part | Who |
|---|---|
| Data collection, pre-processing, exploration, **Task 1** (sections 1–4) | Abdul Sami |
| **Task 2** (section 5) | Fatima |
| Results and discussion (sections 6–7) | both |

## 0. Setup

Install the requirements once from a terminal (we used Python 3.11 on Windows / Anaconda):

```bash
pip install -r requirements.txt
```

Then put the data files in `data/raw/` (see section 1) and run the notebook top to bottom.

In [1]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")   # silences a joblib warning on Windows

import json, random, re, time, warnings
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, adjusted_rand_score, classification_report,
                             confusion_matrix, f1_score, normalized_mutual_info_score)
from sklearn.metrics.pairwise import linear_kernel
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.dpi"] = 110

### Configuration

We kept every setting in one dictionary, so we never had to hunt for numbers inside the functions.

In [ ]:
SMOKE_TEST = False   # True = quick run on a small fake dataset, only to check the installation

CONFIG = {
    "seed": 42,
    "paths": {
        "raw_dir": Path("data/raw"),             # the CSV files of the CFPB narratives archive
        "processed_dir": Path("data/processed"),
        "results_dir": Path("results"),
        "cache_dir": Path("data/processed/cache"),
    },
    "data": {
        "date_from": "2023-01-01", "date_to": "2024-12-31",
        "n_products": 5,       # the 5 most frequent products
        "per_class": 4000,     # 5 x 4,000 = 20,000 complaints before cleaning
        "min_words": 20,       # very short narratives say almost nothing
        "split": {"train": 0.70, "val": 0.15, "test": 0.15},
        "chunksize": 200_000,  # rows per chunk, so the big files fit in memory
        "near_dup_threshold": 0.90,
        # the CFPB renamed some products over the years, so we merge the old and new names
        "product_merge": {
            "Credit reporting, credit repair services, or other personal consumer reports": "Credit reporting",
            "Credit reporting or other personal consumer reports": "Credit reporting",
            "Credit card or prepaid card": "Credit card",
        },
    },
    "preprocess": {"remove_redactions": True, "lemmatize": True, "keep_negations": True},
}

In [ ]:
CONFIG["classification"] = {
    "tfidf": {"ngram_range": (1, 2), "min_df": 3, "max_df": 0.9, "sublinear_tf": True},
    "C_grid": [0.1, 1.0, 10.0],
    "use_embeddings": True,
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "batch_size": 64,
    "zeroshot": {"enabled": False, "model": "facebook/bart-large-mnli", "n_samples": 500},
}

CONFIG["topics"] = {"k_grid": [5, 8, 10, 15, 20], "n_top_words": 10, "lda_max_iter": 20,
                    "use_embeddings": True}

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
    except ImportError:
        pass


def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=float)


def savefig(fig, name):
    path = RES / "figures" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path)
    plt.show()


set_seed(CONFIG["seed"])
for p in CONFIG["paths"].values():
    p.mkdir(parents=True, exist_ok=True)
RES = CONFIG["paths"]["results_dir"]